# Module 1: Why OpenEnv? — Your First Environments

In this notebook you'll connect to three real hosted OpenEnv environments and interact with each using the same 3-method interface: `reset()`, `step()`, `state()`.

**Time:** ~15 min · **Difficulty:** Beginner · **GPU:** Not required

In [8]:
# Install dependencies
!uv pip install -q openenv-core==0.2.2

## 1. The Echo Environment

The simplest possible OpenEnv environment — it echoes back whatever you send. Perfect for learning the interface.

Hosted at: `https://openenv-echo-env.hf.space`

In [ ]:
!uv pip install -q "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/echo_env"

In [10]:
from echo_env import EchoEnv, CallToolAction

# Connect to the hosted Echo environment
with EchoEnv(base_url="https://mroyme-echo-env.hf.space").sync() as env:
    # reset() starts a new episode
    result = env.reset()
    print("After reset:")
    print(f"  Observation: {result.observation}")
    print(f"  Done: {result.done}")
    print()

    # step() takes an action and returns the next observation
    result = env.step(CallToolAction(tool_name="echo_with_length", arguments= {"message": "Hello, OpenEnv!"}))
    print("After step:")
    print(f"  Observation: {result.observation}")
    print(f"  Reward: {result.reward}")
    print(f"  Done: {result.done}")
    print()

    # state() returns episode metadata
    state = env.state()
    print("State:")
    print(f"  Episode ID: {state.episode_id}")
    print(f"  Step count: {state.step_count}")

After reset:
  Observation: done=False reward=0.0 metadata={}
  Done: False

After step:
  Observation: done=False reward=None metadata={} tool_name='echo_with_length' result={'content': [{'type': 'text', 'text': '{"message":"Hello, OpenEnv!","length":15}', 'annotations': None, 'meta': None}], 'structured_content': {'message': 'Hello, OpenEnv!', 'length': 15}, 'meta': None, 'data': {'message': 'Hello, OpenEnv!', 'length': 15}, 'is_error': False} error=None
  Reward: None
  Done: False

State:
  Episode ID: e66853fa-0b7f-4a21-bf73-68b5dcb5fbd9
  Step count: 1


Three methods. That's the entire API. Every OpenEnv environment works exactly like this.

## 2. OpenSpiel Catch

Now let's connect to a real game. Catch is a simple single-player game from DeepMind's OpenSpiel:

- A ball falls from the top of a 10×5 grid
- You move a paddle left/right to catch it
- Actions: `0` = left, `1` = stay, `2` = right
- Reward: `+1` if caught, `0` if missed

Same 3 methods, completely different game.

In [11]:
!uv pip install -q "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/openspiel_env"

In [12]:
from openspiel_env import OpenSpielEnv
from openspiel_env.models import OpenSpielAction

OPENSPIEL_URL = "https://mroyme-openspiel-env.hf.space"

with OpenSpielEnv(base_url=OPENSPIEL_URL).sync() as env:
    result = env.reset()
    print(f"Game: Catch")
    print(f"Legal actions: {result.observation.legal_actions}")
    print(f"Info state length: {len(result.observation.info_state)}")
    print()

    # Play a few steps with a random policy
    import random
    step = 0
    while not result.done:
        action_id = random.choice(result.observation.legal_actions)
        action_name = {0: "LEFT", 1: "STAY", 2: "RIGHT"}[action_id]
        result = env.step(OpenSpielAction(
            action_id=action_id,
            game_name="catch"
        ))
        step += 1
        print(f"Step {step}: {action_name} → reward={result.reward}, done={result.done}")

    print(f"\nFinal reward: {result.reward}")
    print(f"State: {env.state()}")

Game: Catch
Legal actions: [0, 1, 2]
Info state length: 50

Step 1: STAY → reward=0.0, done=False
Step 2: LEFT → reward=0.0, done=False
Step 3: LEFT → reward=0.0, done=False
Step 4: STAY → reward=0.0, done=False
Step 5: LEFT → reward=0.0, done=False
Step 6: LEFT → reward=0.0, done=False
Step 7: RIGHT → reward=0.0, done=False
Step 8: STAY → reward=0.0, done=False
Step 9: STAY → reward=-1.0, done=True

Final reward: -1.0
State: episode_id='d676b015-3b86-41b8-9d74-c5c45411d790' step_count=9 game_name='catch' agent_player=0 opponent_policy='random' game_params={} num_players=1


Same pattern: `reset()` → `step()` → check `done`. The observation type is different (`OpenSpielObservation` vs `EchoObservation`), but the interface is identical.

## 3. TextArena Wordle

TextArena is a text-based game environment. Wordle gives you 6 attempts to guess a 5-letter word, with color-coded feedback after each guess.

Hosted at: `https://burtenshaw-textarena.hf.space`

In [13]:
!uv pip install -q "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/textarena_env"

In [14]:
from textarena_env import TextArenaEnv, TextArenaAction

TEXTARENA_URL = "https://mroyme-textarena-env.hf.space"

with TextArenaEnv(base_url=TEXTARENA_URL).sync() as env:
    result = env.reset()
    print("Wordle prompt:")
    print(result.observation.prompt)
    print()

    # Make a guess
    guesses = ["crane", "slate", "blind"]
    for guess in guesses:
        if result.done:
            break
        result = env.step(TextArenaAction(message=f"[{guess}]"))
        print(f"Guess: {guess}")
        for msg in result.observation.messages:
            print(f"  [{msg.category}] {msg.content}")
        print(f"  Reward: {result.reward}, Done: {result.done}")
        print()

Wordle prompt:
[GAME] You are Playing Wordle.
A secret 5-letter word has been chosen. You have 6 attempts to guess it.
For each guess, wrap your word in square brackets (e.g., '[apple]').
Feedback for each letter will be given as follows:
  - G (green): correct letter in the correct position
  - Y (yellow): letter exists in the word but in the wrong position
  - X (wrong): letter is not in the word
Enter your guess to begin.

Guess: crane
  [MESSAGE] 
[GAME] You are Playing Wordle.
A secret 5-letter word has been chosen. You have 6 attempts to guess it.
For each guess, wrap your word in square brackets (e.g., '[apple]').
Feedback for each letter will be given as follows:
  - G (green): correct letter in the correct position
  - Y (yellow): letter exists in the word but in the wrong position
  - X (wrong): letter is not in the word
Enter your guess to begin.

[GAME] [crane]
[GAME] You submitted [crane].
Feedback:
C R A N E
X X X Y X
You have 5 guesses left.
  Reward: 0.0, Done: False

G

## 4. Async vs Sync

OpenEnv clients are async by default. For notebooks and simple scripts, use the `.sync()` wrapper:

```python
# Sync (notebooks, simple scripts)
with EchoEnv(base_url=url).sync() as env:
    result = env.reset()

# Async (production, training loops)
async with EchoEnv(base_url=url) as env:
    result = await env.reset()
```

For this course, we'll use `.sync()` everywhere for simplicity.

## Summary

You connected to three completely different environments — Echo, Catch, Wordle — using the same interface:

| Method | What it does |
|--------|--------------|
| `reset()` | Start a new episode |
| `step(action)` | Take an action, get observation + reward |
| `state()` | Get episode metadata |

The action and observation types change per environment, but the pattern never does.

**Next:** [Module 2](../module-2/README.md) — Using existing environments to build and compare policies.